In [ ]:
import pandas as pd
import numpy as np
import os
import re
import json
import zipfile

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
from google.colab import files

uploaded = files.upload()

print("\nUploaded files:")
for filename in uploaded.keys():
    print(filename)

Saving data.zip to data.zip

Uploaded files:
data.zip


In [ ]:
import zipfile
import os

zip_file = "data.zip"

with zipfile.ZipFile(zip_file, "r") as z:
    print("Files inside data.zip:\n")
    for name in z.namelist():
        print(name)

Files inside data.zip:

data/
data/archive (7).zip
data/archive (8).zip
data/online+retail.zip


In [ ]:
import zipfile
import os

base_dir = "/content/billing_datasets"
os.makedirs(base_dir, exist_ok=True)

# First, extract the main data.zip file
with zipfile.ZipFile("data.zip", "r") as z:
    z.extractall("/content") # Extract data.zip into /content, which will create /content/data/ directory

# Now, extract the nested zip files from their correct location
# Extract Customer Support
with zipfile.ZipFile("/content/data/archive (7).zip", "r") as z:
    z.extractall(os.path.join(base_dir, "customer_support"))

# Extract Amazon Fashion
with zipfile.ZipFile("/content/data/archive (8).zip", "r") as z:
    z.extractall(os.path.join(base_dir, "amazon_fashion"))

# Extract Online Retail
with zipfile.ZipFile("/content/data/online+retail.zip", "r") as z: # Corrected filename from online-retail.zip to online+retail.zip
    z.extractall(os.path.join(base_dir, "online_retail"))

print("All three datasets extracted successfully.")

All three datasets extracted successfully.


In [ ]:
import os

for root, dirs, files in os.walk("/content/billing_datasets"):
    for file in files:
        print(os.path.join(root, file))

/content/billing_datasets/online_retail/Online Retail.xlsx
/content/billing_datasets/customer_support/customer_support_500_cleaned.csv
/content/billing_datasets/customer_support/Customer Support Data Set - Sheet1 (1).csv
/content/billing_datasets/amazon_fashion/AMAZON_FASHION.json


In [ ]:
import pandas as pd
import numpy as np
import re
import os

support_path = "/content/billing_datasets/customer_support/Customer Support Data Set - Sheet1 (1).csv"

support_df = pd.read_csv(support_path)

print("Dataset Shape:", support_df.shape)
print("\nColumns:")
print(support_df.columns.tolist())

display(support_df.head(10))

Dataset Shape: (500, 3)

Columns:
['question', 'answer', 'category']


,question,answer,category
0,Where is my order?,You can track your order using the tracking li...,Shipping
1,How do I return a shirt that doesn't fit?,You can return any item within 30 days in its ...,Returns
2,"My package hasn't arrived yet, what should I do?",Please check the tracking number. If it hasn't...,Shipping
3,Can I exchange a dress for a different size?,"Yes, use our exchange form online to select th...",Returns
4,Do you offer free shipping?,"Yes, orders over $50 qualify for free standard...",Shipping
5,I entered the wrong shipping address. Can I ch...,Contact our support immediately; we may be abl...,Shipping
6,How long does shipping take?,Standard shipping usually takes 5-7 business d...,Shipping
7,My payment was declined. What should I do?,"Check your card details and try again, or use ...",Payment
8,Can I cancel my order?,Orders can be canceled within 24 hours of plac...,Order Management
9,I received the wrong item. What now?,Please use our returns page to report the erro...,Returns


In [ ]:
print("Missing values:")
print(support_df.isnull().sum())

print("\nDuplicate rows:")
print(support_df.duplicated().sum())

print("\nCategory distribution:")
print(support_df["category"].value_counts())

Missing values:
question    0
answer      0
category    0
dtype: int64

Duplicate rows:
0

Category distribution:
category
Orders                  143
Shipping                106
Payment                  74
Returns                  62
Product Info             37
Account                  33
Sizing                   15
Order Management         11
Support                   6
Student Records           2
IT Support                2
Faculty Support           1
Campus Services           1
IT Access                 1
Academic Information      1
Classroom Support         1
Alumni Support            1
Web Support               1
Accessibility             1
IT Security               1
Name: count, dtype: int64


In [ ]:
irrelevant_categories = [
    "Student Records",
    "IT Support",
    "Faculty Support",
    "Campus Services",
    "IT Access",
    "Academic Information",
    "Classroom Support",
    "Alumni Support",
    "Web Support",
    "Accessibility",
    "IT Security"
]

support_clean = support_df[
    ~support_df["category"].isin(irrelevant_categories)
].copy()

print("Original records:", len(support_df))
print("After removing unrelated records:", len(support_clean))

print("\nRemaining categories:")
print(support_clean["category"].value_counts())

Original records: 500
After removing unrelated records: 487

Remaining categories:
category
Orders              143
Shipping            106
Payment              74
Returns              62
Product Info         37
Account              33
Sizing               15
Order Management     11
Support               6
Name: count, dtype: int64


In [ ]:
!pip -q install ftfy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.6 MB/s eta 0:00:00


In [ ]:
from ftfy import fix_text

support_clean["question"] = (
    support_clean["question"]
    .astype(str)
    .apply(fix_text)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

support_clean["answer"] = (
    support_clean["answer"]
    .astype(str)
    .apply(fix_text)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print("Text encoding and whitespace cleaning completed.")

Text encoding and whitespace cleaning completed.


In [ ]:
billing_pattern = (
    r"\b(refund|refunds|refunded|payment|paid|charge|charged|"
    r"billing|invoice|receipt|discount|coupon|promo|price|"
    r"tax|fee|fees|money)\b"
)

billing_mask = (
    (support_clean["category"] == "Payment") |
    (
        support_clean["question"]
        .str.contains(
            billing_pattern,
            case=False,
            regex=True,
            na=False
        )
    )
)

billing_support = support_clean[billing_mask].copy()

print("Billing-related records:", len(billing_support))

print("\nSource category distribution:")
print(billing_support["category"].value_counts())

display(
    billing_support[
        ["category", "question", "answer"]
    ].head(20)
)

Billing-related records: 87

Source category distribution:
category
Payment     74
Returns      7
Shipping     3
Orders       2
Account      1
Name: count, dtype: int64


/tmp/ipykernel_1333/3987098124.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(


,category,question,answer
7,Payment,My payment was declined. What should I do?,"Check your card details and try again, or use ..."
11,Payment,Is it safe to use my credit card on your site?,"Yes, we use SSL encryption to ensure secure tr..."
19,Payment,How do I use a discount code?,Enter the promo code at checkout in the 'Apply...
26,Payment,My discount code isn't working.,Check the code's validity date and conditions;...
28,Payment,Can I change my billing address?,"Yes, log in and update your billing informatio..."
36,Orders,Can I get a copy of my invoice?,"Yes, download it from your order history."
39,Payment,Can I pay with PayPal?,"Yes, PayPal is available as a payment option a..."
42,Payment,What payment methods do you accept?,"We accept credit/debit cards, PayPal, and Appl..."
43,Payment,Do you offer student discounts?,"Yes, verify your student status at checkout to..."
47,Payment,My promo code expired. Can it still work?,Contact support; we may provide a one-time ext...


In [ ]:
def classify_billing_intent(text):

    text = text.lower()

    if re.search(r"\b(refund|refunded|refunds)\b", text):
        return "REFUND"

    if re.search(r"\b(invoice|receipt)\b", text):
        return "INVOICE_RECEIPT"

    if re.search(
        r"\b(billing address|billing information|billing cycle)\b",
        text
    ):
        return "BILLING_INFORMATION"

    if re.search(
        r"\b(discount|promo|coupon|price matching|gift card|gift cards|"
        r"promotion|promotions|student pricing|student discounts|"
        r"birthday discounts|anniversary|sale item)\b",
        text
    ):
        return "DISCOUNT_PROMOTION"

    if re.search(
        r"\b(payment|payment method|paypal|credit card|"
        r"split payment|installments|pay with|billing payment)\b",
        text
    ):
        return "PAYMENT"

    return "BILLING_OTHER"


billing_support["intent"] = (
    billing_support["question"] + " " +
    billing_support["answer"]
).apply(classify_billing_intent)

print(billing_support["intent"].value_counts())

intent
DISCOUNT_PROMOTION     50
PAYMENT                19
REFUND                  9
BILLING_INFORMATION     6
INVOICE_RECEIPT         3
Name: count, dtype: int64


In [ ]:
before = len(billing_support)

billing_support = billing_support.drop_duplicates(
    subset=["question"],
    keep="first"
).copy()

after = len(billing_support)

print("Records before question deduplication:", before)
print("Records after question deduplication:", after)
print("Duplicate questions removed:", before - after)

Records before question deduplication: 87
Records after question deduplication: 81
Duplicate questions removed: 6


In [ ]:
billing_support["doc_id"] = [
    f"SUP_BILL_{i:04d}"
    for i in range(1, len(billing_support) + 1)
]

billing_support["source"] = "customer_support"

billing_support_final = billing_support[
    [
        "doc_id",
        "source",
        "intent",
        "question",
        "answer"
    ]
].copy()

billing_support_final.columns = [
    "doc_id",
    "source",
    "category",
    "query_text",
    "reference_response"
]

billing_support_final["rating"] = np.nan

print("Final Billing Support Dataset:")
print(billing_support_final.shape)

display(billing_support_final.head())

Final Billing Support Dataset:
(81, 6)


,doc_id,source,category,query_text,reference_response,rating
7,SUP_BILL_0001,customer_support,PAYMENT,My payment was declined. What should I do?,"Check your card details and try again, or use ...",NaN
11,SUP_BILL_0002,customer_support,PAYMENT,Is it safe to use my credit card on your site?,"Yes, we use SSL encryption to ensure secure tr...",NaN
19,SUP_BILL_0003,customer_support,DISCOUNT_PROMOTION,How do I use a discount code?,Enter the promo code at checkout in the 'Apply...,NaN
26,SUP_BILL_0004,customer_support,DISCOUNT_PROMOTION,My discount code isn't working.,Check the code's validity date and conditions;...,NaN
28,SUP_BILL_0005,customer_support,BILLING_INFORMATION,Can I change my billing address?,"Yes, log in and update your billing informatio...",NaN


In [ ]:
support_output = "/content/billing_support_qa.csv"

billing_support_final.to_csv(
    support_output,
    index=False
)

print("Saved:", support_output)

Saved: /content/billing_support_qa.csv


In [ ]:
amazon_path = "/content/billing_datasets/amazon_fashion/AMAZON_FASHION.json"

amazon_df = pd.read_json(
    amazon_path,
    lines=True
)

print("Amazon Fashion Shape:", amazon_df.shape)
print("\nColumns:")
print(amazon_df.columns.tolist())

Amazon Fashion Shape: (883636, 12)

Columns:
['overall', 'verified', 'reviewTime', 'reviewerID', 'asin', 'reviewerName', 'reviewText', 'summary', 'unixReviewTime', 'vote', 'style', 'image']


In [ ]:
import html

amazon_df["reviewText"] = (
    amazon_df["reviewText"]
    .fillna("")
    .astype(str)
)

amazon_df["summary"] = (
    amazon_df["summary"]
    .fillna("")
    .astype(str)
)

def clean_review_text(text):

    text = re.sub(r"<[^>]+>", " ", text)
    text = html.unescape(text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


amazon_df["reviewText_clean"] = (
    amazon_df["reviewText"]
    .apply(clean_review_text)
)

amazon_df["summary_clean"] = (
    amazon_df["summary"]
    .apply(clean_review_text)
)

print("Text cleaning completed.")

Text cleaning completed.


In [ ]:
before = len(amazon_df)

amazon_df = amazon_df[
    amazon_df["reviewText_clean"].str.len() >= 10
].copy()

print("Before:", before)
print("After:", len(amazon_df))
print("Removed:", before - len(amazon_df))

Before: 883636
After: 832241
Removed: 51395


In [ ]:
before = len(amazon_df)

amazon_df = amazon_df.drop_duplicates(
    subset=["reviewText_clean"]
).copy()

print("Before deduplication:", before)
print("After deduplication:", len(amazon_df))
print("Duplicates removed:", before - len(amazon_df))

Before deduplication: 832241
After deduplication: 763135
Duplicates removed: 69106


In [ ]:
amazon_df["combined_text"] = (
    amazon_df["reviewText_clean"] + " " +
    amazon_df["summary_clean"]
).str.lower()

In [ ]:
billing_patterns = {
    "REFUND": r"\b(refund|refunded|refunds)\b",

    "CHARGE_ISSUE": (
        r"\b(overcharged|charged)\b|"
        r"\b(handling|restocking|shipping|return)\s+fees?\b"
    ),

    "PAYMENT_BILLING": (
        r"\b(payment|billing|invoice|paypal)\b"
    ),

    "DISCOUNT_PROMOTION": (
        r"\b(coupon|promo\s*code|discount\s*code)\b"
    )
}

In [ ]:
amazon_df["rating_numeric"] = pd.to_numeric(
    amazon_df["overall"],
    errors="coerce"
)

billing_amazon = amazon_df[
    amazon_df["rating_numeric"] <= 3
].copy()

billing_amazon["billing_category"] = None

for category, pattern in billing_patterns.items():

    mask = (
        billing_amazon["billing_category"].isna() &
        billing_amazon["combined_text"].str.contains(
            pattern,
            regex=True,
            na=False
        )
    )

    billing_amazon.loc[
        mask,
        "billing_category"
    ] = category

billing_amazon = billing_amazon[
    billing_amazon["billing_category"].notna()
].copy()

print(
    billing_amazon["billing_category"]
    .value_counts()
)

/tmp/ipykernel_1333/1269519557.py:16: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  billing_amazon["combined_text"].str.contains(
/tmp/ipykernel_1333/1269519557.py:16: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  billing_amazon["combined_text"].str.contains(
/tmp/ipykernel_1333/1269519557.py:16: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  billing_amazon["combined_text"].str.contains(
/tmp/ipykernel_1333/1269519557.py:16: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  billing_amazon["combined_text"].str.contains(


billing_category
REFUND                3145
CHARGE_ISSUE           320
PAYMENT_BILLING         75
DISCOUNT_PROMOTION      11
Name: count, dtype: int64


In [ ]:
MAX_PER_CATEGORY = 300

billing_amazon_balanced = (
    billing_amazon
    .groupby("billing_category", group_keys=False)
    .apply(
        lambda x: x.sample(
            n=min(len(x), MAX_PER_CATEGORY),
            random_state=42
        )
    )
    .reset_index(drop=True)
)

print(
    billing_amazon_balanced["billing_category"]
    .value_counts()
)

billing_category
CHARGE_ISSUE          300
REFUND                300
PAYMENT_BILLING        75
DISCOUNT_PROMOTION     11
Name: count, dtype: int64


/tmp/ipykernel_1333/1550152717.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [ ]:
billing_amazon_balanced["doc_id"] = [
    f"AMZ_BILL_{i:04d}"
    for i in range(
        1,
        len(billing_amazon_balanced) + 1
    )
]

billing_amazon_balanced["source"] = "amazon_review"

billing_amazon_final = billing_amazon_balanced[
    [
        "doc_id",
        "source",
        "billing_category",
        "reviewText_clean",
        "rating_numeric"
    ]
].copy()

billing_amazon_final.columns = [
    "doc_id",
    "source",
    "category",
    "query_text",
    "rating"
]

billing_amazon_final["reference_response"] = np.nan

billing_amazon_final = billing_amazon_final[
    [
        "doc_id",
        "source",
        "category",
        "query_text",
        "reference_response",
        "rating"
    ]
]

print(billing_amazon_final.shape)
display(billing_amazon_final.head())

(686, 6)


,doc_id,source,category,query_text,reference_response,rating
0,AMZ_BILL_0001,amazon_review,CHARGE_ISSUE,My daughter will not wear this dress. The bead...,NaN,2
1,AMZ_BILL_0002,amazon_review,CHARGE_ISSUE,This should have not been marked an XL. It was...,NaN,2
2,AMZ_BILL_0003,amazon_review,CHARGE_ISSUE,For the price of these you would think they wo...,NaN,3
3,AMZ_BILL_0004,amazon_review,CHARGE_ISSUE,Had to return. Not true to size. Then charged ...,NaN,1
4,AMZ_BILL_0005,amazon_review,CHARGE_ISSUE,This product is much smaller then shown in the...,NaN,2


In [ ]:
amazon_output = "/content/billing_amazon_reviews.csv"

billing_amazon_final.to_csv(
    amazon_output,
    index=False
)

print("Saved:", amazon_output)

Saved: /content/billing_amazon_reviews.csv


In [ ]:
retail_path = "/content/billing_datasets/online_retail/Online Retail.xlsx"

retail_df = pd.read_excel(retail_path)

print("Shape:", retail_df.shape)

print("\nColumns:")
print(retail_df.columns.tolist())

Shape: (541909, 8)

Columns:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


In [ ]:
print("Missing values:")
print(retail_df.isnull().sum())

print("\nDuplicate records:")
print(retail_df.duplicated().sum())

print("\nNegative Quantity:",
      (retail_df["Quantity"] < 0).sum())

print("Zero UnitPrice:",
      (retail_df["UnitPrice"] == 0).sum())

print("Negative UnitPrice:",
      (retail_df["UnitPrice"] < 0).sum())

print(
    "Cancellation invoices:",
    retail_df["InvoiceNo"]
    .astype(str)
    .str.startswith("C")
    .sum()
)

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Duplicate records:
5268

Negative Quantity: 10624
Zero UnitPrice: 2515
Negative UnitPrice: 2
Cancellation invoices: 9288


In [ ]:
duplicate_count = retail_df.duplicated().sum()

retail_df = retail_df.drop_duplicates().copy()

print("Duplicates removed:", duplicate_count)
print("New shape:", retail_df.shape)

Duplicates removed: 5268
New shape: (536641, 8)


In [ ]:
retail_df["InvoiceNo"] = (
    retail_df["InvoiceNo"]
    .astype(str)
    .str.strip()
)

retail_df["StockCode"] = (
    retail_df["StockCode"]
    .astype(str)
    .str.strip()
)

retail_df["Description"] = (
    retail_df["Description"]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.upper()
)

print("Text cleaning completed.")

Text cleaning completed.


In [ ]:
negative_price_count = (
    retail_df["UnitPrice"] < 0
).sum()

retail_df = retail_df[
    retail_df["UnitPrice"] >= 0
].copy()

print(
    "Negative UnitPrice records removed:",
    negative_price_count
)

print(
    "Negative Quantity records retained:",
    (retail_df["Quantity"] < 0).sum()
)

Negative UnitPrice records removed: 2
Negative Quantity records retained: 10587


In [ ]:
retail_df["InvoiceDate"] = pd.to_datetime(
    retail_df["InvoiceDate"]
)

retail_df["LineAmount"] = (
    retail_df["Quantity"] *
    retail_df["UnitPrice"]
).round(2)

retail_df["CustomerID_missing"] = (
    retail_df["CustomerID"].isna()
)

retail_df["TransactionType"] = np.where(
    retail_df["InvoiceNo"].str.startswith("C"),
    "Cancellation",
    np.where(
        retail_df["Quantity"] < 0,
        "Return/Adjustment",
        "Sale"
    )
)

print(
    retail_df["TransactionType"]
    .value_counts()
)

TransactionType
Sale                 526052
Cancellation           9251
Return/Adjustment      1336
Name: count, dtype: int64


In [ ]:
billing_transactions = retail_df[
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "UnitPrice",
        "LineAmount",
        "CustomerID",
        "CustomerID_missing",
        "Country",
        "TransactionType"
    ]
].copy()

print(
    "Final Billing Transaction Shape:",
    billing_transactions.shape
)

print("\nColumns:")
print(billing_transactions.columns.tolist())

Final Billing Transaction Shape: (536639, 11)

Columns:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'LineAmount', 'CustomerID', 'CustomerID_missing', 'Country', 'TransactionType']


In [ ]:
invoice_summary = (
    billing_transactions
    .groupby("InvoiceNo", dropna=False)
    .agg(
        InvoiceDate=("InvoiceDate", "min"),
        CustomerID=("CustomerID", "first"),
        Country=("Country", "first"),
        LineCount=("InvoiceNo", "size"),
        UniqueProducts=("StockCode", "nunique"),
        TotalQuantity=("Quantity", "sum"),
        TotalAmount=("LineAmount", "sum"),
        TransactionType=(
            "TransactionType",
            lambda x:
                "Cancellation"
                if (x == "Cancellation").any()
                else (
                    "Return/Adjustment"
                    if (x == "Return/Adjustment").any()
                    else "Sale"
                )
        )
    )
    .reset_index()
)

invoice_summary["TotalAmount"] = (
    invoice_summary["TotalAmount"]
    .round(2)
)

print(
    "Invoice Summary Shape:",
    invoice_summary.shape
)

display(invoice_summary.head())

Invoice Summary Shape: (25898, 9)


,InvoiceNo,InvoiceDate,CustomerID,Country,LineCount,UniqueProducts,TotalQuantity,TotalAmount,TransactionType
0,536365,2010-12-01 08:26:00,17850.0,United Kingdom,7,7,40,139.12,Sale
1,536366,2010-12-01 08:28:00,17850.0,United Kingdom,2,2,12,22.20,Sale
2,536367,2010-12-01 08:34:00,13047.0,United Kingdom,12,12,83,278.73,Sale
3,536368,2010-12-01 08:34:00,13047.0,United Kingdom,4,4,15,70.05,Sale
4,536369,2010-12-01 08:35:00,13047.0,United Kingdom,1,1,3,17.85,Sale


In [ ]:
transaction_output = (
    "/content/billing_transactions.csv"
)

invoice_output = (
    "/content/billing_invoice_summary.csv"
)

billing_transactions.to_csv(
    transaction_output,
    index=False
)

invoice_summary.to_csv(
    invoice_output,
    index=False
)

print("Saved:")
print(transaction_output)
print(invoice_output)

Saved:
/content/billing_transactions.csv
/content/billing_invoice_summary.csv


In [ ]:
billing_rag = pd.concat(
    [
        billing_support_final,
        billing_amazon_final
    ],
    ignore_index=True
)

print(
    "Billing RAG Corpus Shape:",
    billing_rag.shape
)

print("\nSource distribution:")
print(billing_rag["source"].value_counts())

print("\nCategory distribution:")
print(billing_rag["category"].value_counts())

Billing RAG Corpus Shape: (767, 6)

Source distribution:
source
amazon_review       686
customer_support     81
Name: count, dtype: int64

Category distribution:
category
REFUND                 308
CHARGE_ISSUE           300
PAYMENT_BILLING         75
DISCOUNT_PROMOTION      58
PAYMENT                 18
BILLING_INFORMATION      6
INVOICE_RECEIPT          2
Name: count, dtype: int64


In [ ]:
billing_rag_output = (
    "/content/billing_rag_corpus.csv"
)

billing_rag.to_csv(
    billing_rag_output,
    index=False
)

print(
    "Saved:",
    billing_rag_output
)

Saved: /content/billing_rag_corpus.csv


In [ ]:
print("========== BILLING PREPROCESSING SUMMARY ==========")

print("\n1. Customer Support Q&A")
print("Shape:", billing_support_final.shape)
print(billing_support_final["category"].value_counts())

print("\n2. Amazon Fashion Reviews")
print("Shape:", billing_amazon_final.shape)
print(billing_amazon_final["category"].value_counts())

print("\n3. Billing RAG Corpus")
print("Shape:", billing_rag.shape)
print(billing_rag["source"].value_counts())

print("\n4. Online Retail Transactions")
print("Shape:", billing_transactions.shape)

print("\n5. Invoice Summary")
print("Shape:", invoice_summary.shape)

print("\n========== FILES CREATED ==========")

for file in [
    "/content/billing_support_qa.csv",
    "/content/billing_amazon_reviews.csv",
    "/content/billing_rag_corpus.csv",
    "/content/billing_transactions.csv",
    "/content/billing_invoice_summary.csv"
]:
    print(file, "->", os.path.exists(file))

========== BILLING PREPROCESSING SUMMARY ==========

1. Customer Support Q&A
Shape: (81, 6)
category
DISCOUNT_PROMOTION     47
PAYMENT                18
REFUND                  8
BILLING_INFORMATION     6
INVOICE_RECEIPT         2
Name: count, dtype: int64

2. Amazon Fashion Reviews
Shape: (686, 6)
category
CHARGE_ISSUE          300
REFUND                300
PAYMENT_BILLING        75
DISCOUNT_PROMOTION     11
Name: count, dtype: int64

3. Billing RAG Corpus
Shape: (767, 6)
source
amazon_review       686
customer_support     81
Name: count, dtype: int64

4. Online Retail Transactions
Shape: (536639, 11)

5. Invoice Summary
Shape: (25898, 9)

========== FILES CREATED ==========
/content/billing_support_qa.csv -> True
/content/billing_amazon_reviews.csv -> True
/content/billing_rag_corpus.csv -> True
/content/billing_transactions.csv -> True
/content/billing_invoice_summary.csv -> True


In [ ]:
from google.colab import files

files.download("/content/billing_support_qa.csv")
files.download("/content/billing_amazon_reviews.csv")
files.download("/content/billing_rag_corpus.csv")
files.download("/content/billing_transactions.csv")
files.download("/content/billing_invoice_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>